## Middleware

Middleware provides a way to more tightly control what happens inside the agent. 
Middleware is useful for the following:

1. Tracking agent behavior with logging, analytics, and debugging.
2. Transforming prompts, tool selection, and output formatting.
3. Adding retries, fallbacks, and early termination logic.
4. Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

llm=ChatGroq(
    model='llama-3.3-70b-versatile'
)

agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [4]:

### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='dd2042ef-5200-47fd-ab80-1676dcfb97da'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.013567542, 'completion_tokens_details': None, 'prompt_time': 0.001187739, 'prompt_tokens_details': None, 'queue_time': 0.163281347, 'total_time': 0.014755281}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f99cd-8726-7e91-b10b-1351c781ae36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='dd2042ef-5200-47fd-ab80-1676dcfb97da'), AIMess

### Token Size

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [7]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~137 tokens, 6 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='8bd02896-a67f-403a-98de-a42647690abf'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ys50fpxbv', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.033847625, 'completion_tokens_details': None, 'prompt_time': 0.011671126, 'prompt_tokens_details': None, 'queue_time': 0.058284978, 'total_time': 0.045518751}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f99cf-d0f4-79a2-aa7e-e2a943c21fcb-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'ys50fpxbv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

### Fraction

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~91 tokens (0.0711%), 6 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='0441e889-11cf-455d-84fa-461fe437d922'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bkdmz2z6a', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.050116085, 'completion_tokens_details': None, 'prompt_time': 0.01040719, 'prompt_tokens_details': None, 'queue_time': 0.162840366, 'total_time': 0.060523275}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f99d4-4e06-7b43-b07f-419b120cc5ef-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'bkdmz2z6a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

### Human in the loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

1. High-stakes operations requiring human approval (e.g. database writes, financial transactions).
2. Compliance workflows where human oversight is mandatory.
3. Long-running conversations where human feedback guides the agent.

In [3]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str)-> str:
    """mock function to read all the email by its id"""
    return f"Email content for the id: {email_id}"


def send_email_tool(recipient:str, subject:str, body: str)-> str:
    "mock function to send an email"
    return f"email sent to {recipient} with subject{subject}"



In [6]:
agent =create_agent(
    model=llm,
    tools={read_email_tool,send_email_tool},
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ['approve','edit','reject']
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [7]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [9]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4bc8dbcb-e14c-4cf8-ad41-54f9eb026618'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'b1c6p37m3', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 311, 'total_tokens': 343, 'completion_time': 0.042570436, 'completion_tokens_details': None, 'prompt_time': 0.019130396, 'prompt_tokens_details': None, 'queue_time': 0.052390528, 'total_time': 0.061700832}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa241-b52d-78e2-965e-77c7aec902f8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

In [10]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I hope the email was sent successfully. Let me know if you need to send another email or perform any other task.


In [11]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4bc8dbcb-e14c-4cf8-ad41-54f9eb026618'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'b1c6p37m3', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 311, 'total_tokens': 343, 'completion_time': 0.042570436, 'completion_tokens_details': None, 'prompt_time': 0.019130396, 'prompt_tokens_details': None, 'queue_time': 0.052390528, 'total_time': 0.061700832}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa241-b52d-78e2-965e-77c7aec902f8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b